# Backend-6: 테스트 지연 제품 리스트 (pd_worst / non_pd_worst)

사양서(2026-01-29 추가 백엔드-6 설계)에 따라 **prod_day=20260130** 기준으로
- **day**: 08:30:00 ~ 20:29:59
- **night**: 20:30:00 ~ D+1 08:29:59

두 개 DataFrame을 생성해 출력합니다.

출력 컬럼 순서:
`prod_day, shift_type, barcode_information, pn, remark, station, end_day, end_time, run_time, test_contents, file_path, updated_at`


In [1]:
# !pip -q install sqlalchemy psycopg2-binary pandas

from __future__ import annotations

from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import Tuple

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# =========================
# 1) DB 설정 (사양서)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

# =========================
# 2) 대상 스키마/테이블 (사양서)
# =========================
SRC_SCHEMA = "e4_predictive_maintenance"
T_PD  = "pd_worst"
T_NPD = "non_pd_worst"

MAP_SCHEMA = "g_production_film"
MAP_TABLE  = "remark_info"   # key=barcode_information(18th char)

# =========================
# 3) 고정 실행 조건
# =========================
PROD_DAY = "20260130"  # 요구: 20260130 day & 20260130 night 2개 고정


In [2]:
def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # notebook 단발 실행: 풀 최소화 + 끊김 대비
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )


def parse_prod_day(prod_day: str) -> date:
    # "YYYYMMDD" -> date
    y = int(prod_day[0:4])
    m = int(prod_day[4:6])
    d = int(prod_day[6:8])
    return date(y, m, d)


def shift_windows_kst(prod_day: str) -> Tuple[Tuple[datetime, datetime], Tuple[datetime, datetime]]:
    """사양서 기준 KST window 반환 (포함 구간)."""
    d0 = parse_prod_day(prod_day)
    # day: D 08:30:00 ~ D 20:29:59
    day_start = datetime.combine(d0, time(8, 30, 0), tzinfo=KST)
    day_end   = datetime.combine(d0, time(20, 29, 59), tzinfo=KST)
    # night: D 20:30:00 ~ D+1 08:29:59
    night_start = datetime.combine(d0, time(20, 30, 0), tzinfo=KST)
    night_end   = datetime.combine(d0 + timedelta(days=1), time(8, 29, 59), tzinfo=KST)
    return (day_start, day_end), (night_start, night_end)


engine = make_engine()
(day_start, day_end), (night_start, night_end) = shift_windows_kst(PROD_DAY)

print("DAY  :", day_start, "~", day_end)
print("NIGHT:", night_start, "~", night_end)


DAY  : 2026-01-30 08:30:00+09:00 ~ 2026-01-30 20:29:59+09:00
NIGHT: 2026-01-30 20:30:00+09:00 ~ 2026-01-31 08:29:59+09:00


In [3]:
SQL_TEMPLATE = """
WITH base AS (
    SELECT
        t.barcode_information,
        t.station,
        t.end_day,
        t.end_time,
        t.run_time,
        t.test_contents,
        t.file_path,
        -- 방식 B: end_ts = end_day + end_time (text -> time 캐스팅)
        (t.end_day::timestamp + (t.end_time::time)) AS end_ts,
        substring(t.barcode_information from 18 for 1) AS key18
    FROM {schema}.{table} t
),
mapped AS (
    SELECT
        b.*,
        ri.pn AS mapped_pn,
        ri.remark AS mapped_remark
    FROM base b
    LEFT JOIN {map_schema}.{map_table} ri
      ON ri.barcode_information = b.key18
)
SELECT
    :prod_day AS prod_day,
    :shift_type AS shift_type,
    barcode_information,
    COALESCE(mapped_pn, 'Unknown') AS pn,
    COALESCE(mapped_remark, 'Unknown') AS remark,
    station,
    end_day,
    end_time,
    run_time,
    test_contents,
    file_path,
    :updated_at AS updated_at
FROM mapped
WHERE end_ts BETWEEN :start_ts AND :end_ts
ORDER BY end_ts ASC
"""


def fetch_worst_df(prod_day: str, shift_type: str, start_ts: datetime, end_ts: datetime) -> pd.DataFrame:
    """pd_worst + non_pd_worst를 모두 조회 후 concat."""
    updated_at = datetime.now(KST)  # 스냅샷 시간 (KST aware)
    params = {
        "prod_day": prod_day,
        "shift_type": shift_type,
        "start_ts": start_ts.replace(tzinfo=None),  # DB가 local timestamp라고 가정(사양서 취지)
        "end_ts": end_ts.replace(tzinfo=None),
        "updated_at": updated_at,
    }

    dfs = []
    for tbl in (T_PD, T_NPD):
        sql = SQL_TEMPLATE.format(
            schema=SRC_SCHEMA, table=tbl,
            map_schema=MAP_SCHEMA, map_table=MAP_TABLE,
        )
        with engine.connect() as conn:
            df = pd.read_sql(text(sql), conn, params=params)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    # 컬럼 순서 고정 (요구사항)
    cols = [
        "prod_day", "shift_type", "barcode_information", "pn", "remark", "station",
        "end_day", "end_time", "run_time", "test_contents", "file_path", "updated_at"
    ]
    out = out[cols]

    return out


df_day   = fetch_worst_df(PROD_DAY, "day",   day_start, day_end)
df_night = fetch_worst_df(PROD_DAY, "night", night_start, night_end)

print("DAY rows  :", len(df_day))
print("NIGHT rows:", len(df_night))


C:\Users\user\AppData\Local\Temp\ipykernel_13612\4085895686.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(dfs, ignore_index=True)


DAY rows  : 5
NIGHT rows: 14


C:\Users\user\AppData\Local\Temp\ipykernel_13612\4085895686.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(dfs, ignore_index=True)


In [4]:
# 출력 (사양: 주간/야간 별로 dataframe 출력)
df_day


,prod_day,shift_type,barcode_information,pn,remark,station,end_day,end_time,run_time,test_contents,file_path,updated_at
0,20260130,day,BA1WJ26030500685SJ8T-14F014-AF,35930927,PD,FCT1,2026-01-30,08:56:51,47.1,1.00_load_c_cc_rng_set,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:53.014296+00:00
1,20260130,day,BA1WJ26030500003USJ8T-14F014-AF,35930928,PD,FCT1,2026-01-30,16:42:49,47.3,1.15_load_c_rng_set,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:53.014296+00:00
2,20260130,day,BA1WJ26028500025SJ8T-14F014-AF,35930927,PD,FCT1,2026-01-30,17:07:38,52.6,1.28_load_c_3c_set,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:53.014296+00:00
3,20260130,day,BA1WJ26028500158SJ8T-14F014-AF,35930927,PD,FCT1,2026-01-30,17:37:38,45.5,1.25_overcurr_usb_c_c,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:53.014296+00:00
4,20260130,day,BA1WJ26030501405USJ8T-14F014-AF,35930928,PD,FCT1,2026-01-30,19:46:20,54.3,1.27_usb_a_v,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:53.014296+00:00


In [5]:
df_night


,prod_day,shift_type,barcode_information,pn,remark,station,end_day,end_time,run_time,test_contents,file_path,updated_at
0,20260130,night,BA1WJ26031501148USJ8T-14F014-AF,35930928,PD,FCT1,2026-01-31,04:13:01,46.6,1.18_load_c_sensing_on,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
1,20260130,night,BA1WJ26031501082USJ8T-14F014-AF,35930928,PD,FCT1,2026-01-31,04:35:06,41.5,1.21_no_load_usb_a,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
2,20260130,night,BA1WJ26031500875USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,04:39:57,41.6,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
3,20260130,night,BA1WJ26031501206USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,04:55:31,41.6,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
4,20260130,night,BA1WJ26031501315USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,04:59:00,47.5,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
5,20260130,night,BA1WJ26031501347USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,05:14:27,41.7,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
6,20260130,night,BA1WJ26031501266USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,05:25:29,47.7,1.26_overcurr_usb_c_v,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
7,20260130,night,BA1WJ26031501523USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,05:28:29,41.5,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
8,20260130,night,BA1WJ26031501566USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,05:35:40,41.5,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00
9,20260130,night,BA1WJ26031501585USJ8T-14F014-AF,35930928,PD,FCT3,2026-01-31,05:39:00,41.8,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2026-01-31 07:19:55.958905+00:00


## (추가) DataFrame 저장 (PostgreSQL)

- 주간(day)  → 스키마: `i_daily_report`, 테이블: `f_worst_case_day_daily`
- 야간(night) → 스키마: `i_daily_report`, 테이블: `f_worst_case_night_daily`

동일 `prod_day` 데이터가 이미 존재하면 **해당 prod_day만 삭제 후 재삽입**합니다(스냅샷 교체).


In [6]:
from sqlalchemy import text
from sqlalchemy.types import Text, TIMESTAMP


SAVE_SCHEMA = "i_daily_report"
T_SAVE_DAY = "f_worst_case_day_daily"
T_SAVE_NIGHT = "f_worst_case_night_daily"

SAVE_COLS = [
    "prod_day", "shift_type", "barcode_information", "pn", "remark", "station",
    "end_day", "end_time", "run_time", "test_contents", "file_path", "updated_at"
]

DTYPE_MAP = {
    "prod_day": Text(),
    "shift_type": Text(),
    "barcode_information": Text(),
    "pn": Text(),
    "remark": Text(),
    "station": Text(),
    "end_day": Text(),
    "end_time": Text(),
    "run_time": Text(),
    "test_contents": Text(),
    "file_path": Text(),
    "updated_at": TIMESTAMP(timezone=True),
}


def ensure_schema(engine, schema: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema};"))


def ensure_table(engine, schema: str, table: str) -> None:
    # all TEXT, updated_at timestamptz (사양과 호환, 단발 노트북 저장용)
    cols_ddl = []
    for c in SAVE_COLS:
        if c == "updated_at":
            cols_ddl.append(f'"{c}" timestamptz')
        else:
            cols_ddl.append(f'"{c}" text')
    cols_sql = ",\n  ".join(cols_ddl)

    ddl = f"""
    CREATE TABLE IF NOT EXISTS {schema}.{table} (
      {cols_sql}
    );
    """
    with engine.begin() as conn:
        conn.execute(text(ddl))


def snapshot_replace(engine, schema: str, table: str, prod_day: str, df: pd.DataFrame) -> None:
    # 1) 스키마/테이블 보장
    ensure_schema(engine, schema)
    ensure_table(engine, schema, table)

    # 2) 타입 정리 (end_day는 date일 수 있어 text로 저장)
    dfx = df.copy()
    dfx = dfx[SAVE_COLS]

    # end_day/text 통일
    dfx["end_day"] = dfx["end_day"].astype(str)

    # updated_at은 tz-aware여야 함(KST). tz 정보가 없으면 KST 부여.
    if pd.api.types.is_datetime64_any_dtype(dfx["updated_at"]):
        if getattr(dfx["updated_at"].dt, "tz", None) is None:
            dfx["updated_at"] = dfx["updated_at"].dt.tz_localize(KST)
    else:
        # 혹시 string이면 파싱
        dfx["updated_at"] = pd.to_datetime(dfx["updated_at"], errors="coerce")
        dfx["updated_at"] = dfx["updated_at"].dt.tz_localize(KST)

    # 3) 동일 prod_day 스냅샷 교체(삭제 후 삽입)
    with engine.begin() as conn:
        conn.execute(
            text(f"DELETE FROM {schema}.{table} WHERE prod_day = :prod_day;"),
            {"prod_day": prod_day},
        )

    # 4) 삽입
    dfx.to_sql(
        table,
        engine,
        schema=schema,
        if_exists="append",
        index=False,
        dtype=DTYPE_MAP,
        method="multi",
        chunksize=2000,
    )


# 실행: day / night 저장
snapshot_replace(engine, SAVE_SCHEMA, T_SAVE_DAY, PROD_DAY, df_day)
snapshot_replace(engine, SAVE_SCHEMA, T_SAVE_NIGHT, PROD_DAY, df_night)

print("[OK] saved:", f"{SAVE_SCHEMA}.{T_SAVE_DAY}", "rows=", len(df_day))
print("[OK] saved:", f"{SAVE_SCHEMA}.{T_SAVE_NIGHT}", "rows=", len(df_night))


[OK] saved: i_daily_report.f_worst_case_day_daily rows= 5
[OK] saved: i_daily_report.f_worst_case_night_daily rows= 14
